# Kvasir-VQA x1 — Retrieval-augmented BLIP-2 (evaluation)

Lightweight RAG-style evaluation: retrieve similar samples by question text, format a context prompt, and run BLIP-2 (zero/few-shot) without fine-tuning. This is an evaluation notebook, not training.

In [ ]:

from pathlib import Path
import json
import random
from typing import List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from PIL import Image

import torch
from transformers import Blip2Processor, Blip2ForConditionalGeneration

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:

# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "09_rag_blip2_eval" / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Salesforce/blip2-opt-2.7b"  # swap to lighter if needed
MAX_GEN_TOKENS = 16
RETRIEVE_K = 3
TOP_K_ANSWERS = 20  # restrict retrieval index to common answers to keep context clean
MAX_EVAL_SAMPLES = 200  # limit test set for quick eval; set None for full

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


In [ ]:

# Load metadata
meta = pd.read_csv(META_CSV)
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta = meta.dropna(subset=["question", "answer", "image_path"]).reset_index(drop=True)
meta = meta[meta["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)

answer_counts = meta["answer"].value_counts()
top_answers = set(answer_counts.head(TOP_K_ANSWERS).index)
index_df = meta[meta["answer"].isin(top_answers)].reset_index(drop=True)

# Build splits
train_df = index_df[index_df["split"] == "train"].reset_index(drop=True)
val_df   = index_df[index_df["split"] == "validation"].reset_index(drop=True)
test_df  = index_df[index_df["split"] == "test"].reset_index(drop=True)

if MAX_EVAL_SAMPLES:
    test_df = test_df.sample(min(MAX_EVAL_SAMPLES, len(test_df)), random_state=SEED).reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


In [ ]:

# Build retriever on questions (TF-IDF + cosine NN)
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
train_qs = train_df["question"].tolist()
vec_train = vectorizer.fit_transform(train_qs)

nn = NearestNeighbors(n_neighbors=RETRIEVE_K, metric="cosine")
nn.fit(vec_train)

print("Retriever fitted on", len(train_qs), "questions")


In [ ]:

# Load BLIP-2
processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"
model.config.use_cache = False


In [ ]:
# Retrieval helper
def retrieve_context(question: str) -> List[str]:
    vec = vectorizer.transform([question])
    dist, idx = nn.kneighbors(vec, n_neighbors=RETRIEVE_K)
    rows = train_df.iloc[idx[0]]
    lines = [f"Q: {r['question']} A: {r['answer']}" for _, r in rows.iterrows()]
    return lines

PROMPT_TEMPLATE = "You are a medical VQA assistant. Use the retrieved examples and the image to answer the question concisely.\nExamples:\n{examples}\nQuestion: {question}\nAnswer:"

def generate_answer(row):
    img = Image.open(row["image_path"]).convert("RGB")
    ctx = "\n".join(retrieve_context(row["question"]))
    prompt = PROMPT_TEMPLATE.format(examples=ctx, question=row["question"])
    inputs = processor(images=img, text=prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_GEN_TOKENS)
    return processor.tokenizer.decode(out[0], skip_special_tokens=True).strip()

In [ ]:

# Evaluate on test subset
preds = []
refs = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="RAG BLIP2 eval"):
    preds.append(generate_answer(row))
    refs.append(str(row["answer"]))

results = {"pred": preds, "ref": refs}
pd.DataFrame(results).to_csv(OUT_DIR / "predictions.csv", index=False)

# Simple metrics: exact match rate on top-K answers
exact = [p.lower() == r.lower() for p, r in zip(preds, refs)]
acc = sum(exact) / len(exact) if exact else 0

metrics = {"exact_match": acc, "n": len(test_df)}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)
